# trajectreview modeling notebook
この notebook は `DA3Metric-Large` による metric depth 推定と、`ARCore pose` / intrinsics による world projection を行います。


In [ ]:
CONFIG = {
    'session_root': '/content/drive/MyDrive/trajectreview/input/replace-session-id',
    'work_root': '/content/trajectreview',
    'result_root': '/content/drive/MyDrive/trajectreview/results',
}
CONFIG


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json

session_root = Path(CONFIG['session_root'])
session_package = json.loads((session_root / 'session_package.json').read_text(encoding='utf-8'))
camera_calibration = json.loads((session_root / 'camera_calibration_summary.json').read_text(encoding='utf-8'))
session_id = session_package['sessionId']
selected_route_path = session_root / 'selected_route.json'
job_request_path = session_root / 'colab_job_request.json'
route_id = 'route-da3metric-large-10fps-per-frame-intrinsics'
sampling_profile = '10fps'
intrinsics_mode = 'per_frame'
if selected_route_path.exists():
    selected_route = json.loads(selected_route_path.read_text(encoding='utf-8'))
    route_id = selected_route.get('selectedRouteId', route_id)
    selected_payload = selected_route.get('selectedRoute', {})
    sampling_profile = selected_payload.get('samplingProfile', sampling_profile)
    intrinsics_mode = selected_payload.get('intrinsicsMode', intrinsics_mode)
elif job_request_path.exists():
    job_request = json.loads(job_request_path.read_text(encoding='utf-8'))
    route_id = job_request.get('defaultRouteId', route_id)
images_source = session_root / 'trajectreview' / 'image'
required = [
    session_root / 'video.mp4',
    session_root / 'session_package.json',
    session_root / 'frame_pose_index.csv',
    session_root / 'camera_calibration_summary.json',
    session_root / 'sensor_quality.json',
    session_root / 'space_handoff_manifest.json',
    images_source,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'missing inputs: {missing}')
camera_calibration.get('imageIntrinsicsCoverageRatio'), camera_calibration.get('lensDistortionCoverageRatio')
session_root, session_id, route_id, sampling_profile, intrinsics_mode


In [ ]:
import subprocess

def run(cmd):
    print('RUN', ' '.join(cmd))
    subprocess.run(cmd, check=True)

run(['bash', '-lc', 'apt-get update'])
run(['bash', '-lc', 'apt-get install -y ffmpeg git'])
run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'])
run(['git', 'clone', 'https://github.com/ByteDance-Seed/depth-anything-3.git'])
run(['python', '-m', 'pip', 'install', '-e', './depth-anything-3'])


In [ ]:
from pathlib import Path
work_root = Path(CONFIG['work_root'])
images_dir = work_root / 'image'
sampled_dir = work_root / 'sampled_images'
da3_export_dir = work_root / 'da3_output'
export_dir = Path(CONFIG['result_root']) / session_id / route_id
for directory in [work_root, images_dir, sampled_dir, da3_export_dir, export_dir]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import shutil
for image_path in images_source.iterdir():
    if image_path.is_file():
        shutil.copy2(image_path, images_dir / image_path.name)
stride = 2 if sampling_profile == '5fps' else 1
for index, image_path in enumerate(sorted(images_dir.iterdir())):
    if image_path.is_file() and index % stride == 0:
        shutil.copy2(image_path, sampled_dir / image_path.name)
print({'copied_images': len(list(images_dir.iterdir())), 'sampled_images': len(list(sampled_dir.iterdir()))})


In [ ]:
run([
    'da3', 'auto', str(sampled_dir),
    '--export-format', 'ply',
    '--export-dir', str(da3_export_dir),
    '--model-dir', 'depth-anything/da3metric-large',
])


In [ ]:
summary = {
    'sessionId': session_id,
    'routeId': route_id,
    'samplingProfile': sampling_profile,
    'intrinsicsMode': intrinsics_mode,
    'status': 'completed',
}
(export_dir / 'remote_summary.json').write_text(__import__('json').dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
